In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("dipankarsrirag/topic-modelling-on-emails")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'topic-modelling-on-emails' dataset.
Path to dataset files: /kaggle/input/topic-modelling-on-emails


In [ ]:
import shutil

# ضغط المجلد
shutil.make_archive('dataset', 'zip', path)

# تحميل الملف المضغوط
from google.colab import files
files.download('dataset.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import os

files = os.listdir(path)
print(files)

['Data']


In [ ]:
data_path = path + "/Data"

files = os.listdir(data_path)
print(files)

['Politics', 'Crime', 'Science', 'Entertainment']


In [ ]:
import os
import pandas as pd

texts = []
labels = []

for label in os.listdir(data_path):
    folder_path = os.path.join(data_path, label)

    if os.path.isdir(folder_path):
        for file in os.listdir(folder_path):
            file_path = os.path.join(folder_path, file)

            try:
                with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
                    texts.append(f.read())
                    labels.append(label)
            except:
                continue

df = pd.DataFrame({
    'text': texts,
    'label': labels
})

print(df['label'].value_counts())

label
Science          4000
Politics         3001
Crime            1100
Entertainment    1053
Name: count, dtype: int64


In [ ]:
print("عدد البيانات:", len(df))

عدد البيانات: 9154


In [ ]:
df.dropna(inplace=True)

In [ ]:
print("عدد البيانات:", len(df))

عدد البيانات: 9154


In [ ]:
import re

def clean_text(text):
    text = text.lower()
    text = re.sub(r'\W', ' ', text)  # حذف الرموز
    text = re.sub(r'\d', '', text)   # حذف الأرقام
    text = re.sub(r'\s+', ' ', text) # إزالة الفراغات الزائدة
    return text

df['text'] = df['text'].apply(clean_text)

In [ ]:
#preprocessing
df['text'] = df['text'].str.lower()


In [ ]:
df['label'].value_counts()

,count
label,
Science,4000
Politics,3001
Crime,1100
Entertainment,1053


In [ ]:
import re

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'\W', ' ', text)
    text = re.sub(r'\d+', '', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

df['text'] = df['text'].apply(clean_text)

In [ ]:
print (df.head())

                                                text     label
0  in article philcht h netcom com phil netcom co...  Politics
1  sender usenet martha utcc utk edu usenet news ...  Politics
2  from bc cleveland freenet edu mark ira kaufman...  Politics
3  the fbi released large amounts of cs tear gas ...  Politics
4  to margoli watson ibm com larry margolis from ...  Politics


In [ ]:
# =========================
# تجربة عدة خوارزميات على نفس البيانات الحالية
# =========================

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report

from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier

# النصوص والتصنيفات من نفس البيانات
from sklearn.feature_extraction.text import TfidfVectorizer

# النصوص والتصنيفات
X = df['text']
y = df['label']

# تحويل النصوص إلى TF-IDF
vectorizer = TfidfVectorizer(
    max_features=15000,
    ngram_range=(1,3),
    stop_words='english',
    min_df=3,
    max_df=0.85
)

X_tfidf = vectorizer.fit_transform(X)

print(X_tfidf.shape)
# تثبيت المكتبة إذا لم تكن موجودة
!pip install imbalanced-learn

from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, accuracy_score

# موازنة البيانات
smote = SMOTE(random_state=42)

X_balanced, y_balanced = smote.fit_resample(X_tfidf, y)

# التحقق من التوازن
import pandas as pd
print(pd.Series(y_balanced).value_counts())

# تقسيم البيانات بعد الموازنة
X_train, X_test, y_train, y_test = train_test_split(
    X_balanced,
    y_balanced,
    test_size=0.2,
    random_state=42,
    stratify=y_balanced
)


# النماذج
models = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000,
        class_weight='balanced'
    ),

    "SVM": LinearSVC(
        class_weight='balanced'
    ),

    "Naive Bayes": MultinomialNB(),



}

# التجريب
results = {}

for name, model in models.items():
    print("=" * 50)
    print(f"Training {name}")

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    results[name] = acc

    print(f"{name} Accuracy: {acc:.4f}")
    print(classification_report(y_test, y_pred))

# أفضل نموذج
print("\n" + "=" * 50)
print("Final Results")

for model_name, score in results.items():
    print(f"{model_name}: {score:.4f}")

best_model = max(results, key=results.get)
print(f"\nBest Model: {best_model}")
print(f"Best Accuracy: {results[best_model]:.4f}")

(9154, 15000)
label
Politics         4000
Crime            4000
Science          4000
Entertainment    4000
Name: count, dtype: int64
Training Logistic Regression
Logistic Regression Accuracy: 0.6256
               precision    recall  f1-score   support

        Crime       0.38      0.43      0.41       800
Entertainment       0.38      0.43      0.41       800
     Politics       0.98      0.96      0.97       800
      Science       0.90      0.67      0.77       800

     accuracy                           0.63      3200
    macro avg       0.66      0.63      0.64      3200
 weighted avg       0.66      0.63      0.64      3200

Training SVM
SVM Accuracy: 0.6278
               precision    recall  f1-score   support

        Crime       0.38      0.45      0.41       800
Entertainment       0.38      0.43      0.40       800
     Politics       0.99      0.97      0.98       800
      Science       0.94      0.66      0.78       800

     accuracy                           0.63  

In [ ]:
print(df['label'].value_counts())

label
Science          4000
Politics         3001
Crime            1100
Entertainment    1053
Name: count, dtype: int64


In [ ]:
for label in os.listdir(data_path):
    folder = os.path.join(data_path, label)
    print(label, "عدد الملفات:", len(os.listdir(folder)))

Entertainment عدد الملفات: 1053
Crime عدد الملفات: 1100
Politics عدد الملفات: 3001
Science عدد الملفات: 4001


In [ ]:
from collections import Counter
import re

# دمج كل النصوص في نص واحد
all_text = " ".join(df['text'].astype(str))

# تنظيف بسيط
all_text = all_text.lower()
all_text = re.sub(r'\W', ' ', all_text)

# تقسيم إلى كلمات
words = all_text.split()

# حساب التكرار
word_counts = Counter(words)

# عرض أكثر 20 كلمة
common_words = word_counts.most_common(20)

print("Most Common Words:")
for word, count in common_words:
    print(f"{word}: {count}")

Most Common Words:
the: 149644
to: 75778
of: 69967
a: 58849
and: 58039
in: 48375
that: 39250
is: 39107
i: 38437
it: 30700
for: 24991
you: 23918
be: 20284
this: 19158
on: 18725
s: 18329
are: 18038
not: 17194
have: 16422
t: 15564


In [ ]:
import numpy as np
import pandas as pd

# استخراج أسماء الكلمات من TF-IDF
feature_names = vectorizer.get_feature_names_out()

# حساب متوسط الوزن لكل كلمة
mean_tfidf = X_tfidf.mean(axis=0).A1

# ترتيب الكلمات حسب الأهمية
top_indices = np.argsort(mean_tfidf)[::-1][:20]

important_words = pd.DataFrame({
    'word': feature_names[top_indices],
    'score': mean_tfidf[top_indices]
})

print(important_words)

          word     score
0          edu  0.041168
1          com  0.033455
2       writes  0.022070
3      article  0.020111
4       people  0.019229
5          apr  0.017729
6          key  0.017650
7         just  0.017527
8          don  0.017005
9         like  0.016439
10        know  0.015821
11  government  0.015620
12         use  0.014382
13       think  0.014018
14        chip  0.012602
15        does  0.012452
16     message  0.012191
17     posting  0.012171
18        time  0.012045
19  encryption  0.012029
